**Guess**

1. Will the transformer beat your AvgEmb (validation F1 0.906 to 0.920)?  By how much?
   * Yes it will beat by 0.9440
2. Which will have the lower best validation loss, AvgEmb (about 0.073) or the transformer?
   * transformer
3. If we REMOVE the position vectors, will F1 go up, go down or stay about the same?  Why?
   * if we remove the position vectors, F1 will go down because positions give the order. Without them the model is a bag of words, and “call now” and “now call” look the same.
4. In one sentence: when would a transformer clearly beat TF-IDF + logistic regression?
   * it will beat when it comes to understanding complex context and word order, lots of data, or a model that is pretrained on a huge corpus.

In [1]:
import re, copy, numpy as np, pandas as pd, torch, torch.nn as nn
from collections import Counter
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

In [2]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "text"])
df["y"] = (df["label"] == "spam").astype(int)
print(df.shape)
print(df["y"].value_counts(), df["y"].mean())
print(df.sample(5, random_state=1))

(5572, 3)
y
0    4825
1     747
Name: count, dtype: int64 0.13406317300789664
     label                                               text  y
1078   ham                       Yep, by the pretty sculpture  0
4028   ham      Yes, princess. Are you going to make me moan?  0
958    ham                         Welp apparently he retired  0
4642   ham                                            Havent.  0
4674   ham  I forgot 2 ask ü all smth.. There's a card on ...  0


In [3]:
base_acc = 1 - df["y"].mean()
print("baseline acc:", round(base_acc, 4))

baseline acc: 0.8659


In [4]:
X_tr_all, X_te, y_tr_all, y_tr = train_test_split(
    df["text"], df["y"], test_size=0.2, random_state=42, stratify=df["y"]
)

X_tr, X_va, y_tr, y_va = train_test_split(
    X_tr_all, y_tr_all, test_size=0.2, random_state=42, stratify=y_tr_all
)

print(len(X_tr_all),  len(X_te), len(X_tr), len(X_va), int(y_va.sum()))

4457 1115 3565 892 120


In [5]:
def tokenise(t):
    return re.findall(r"[a-z0-9']+", t.lower())

In [6]:
cnt = Counter(w for t in X_tr for w in tokenise(t))
itos = ["<PAD>", "<UNK>"] + sorted( w for w, c in cnt.items() if c >= 2)
stoi = {w: i for i, w in enumerate(itos)}
print("vocab size", len(itos), "of", len(cnt), "distinct training words")

vocab size 3332 of 6981 distinct training words


In [7]:
lens = X_tr.map(lambda t: len(tokenise(t)))
print(round(lens.mean(),  4), np.percentile(lens, 95), lens.max())

15.7332 33.0 121


In [35]:
L = 40
def encode(t):
    ids = [stoi.get(w, 1) for w in tokenise(t)[:L]] or [1]
    return ids + [0] * (L - len(ids))

def to_tensors(X, y):
    return (torch.tensor([encode(t) for t in X]),
            torch.tensor(y.values, dtype=torch.float32).unsqueeze(1))

In [36]:
Xt, yt = to_tensors(X_tr, y_tr)
Xv, yv = to_tensors(X_va, y_va)
print(Xt.shape, yt.shape, Xt[0][:8])

torch.Size([3565, 40]) torch.Size([3565, 1]) tensor([2855, 2755, 2647, 3184,  977, 3314, 1338, 2219])


In [37]:
import math
class SelfAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.d = d

    def forward(self, x, pad):
        Q, K, V = self.q(x), self.k(x), self.v(x)
        S = Q @ K.transpose(1, 2) / math.sqrt(self.d)
        S = S.masked_fill(pad.unsqueeze(1), float("-inf"))
        A = torch.softmax(S, dim=-1)
        self.last = A
        return A @ V

In [38]:
sa = SelfAttention(2)
with torch.no_grad():
    sa.q.weight.copy_(torch.tensor([[1,1], [0,1]], dtype=torch.float32).T)
    sa.k.weight.copy_(torch.tensor([[1,0], [1,1]], dtype=torch.float32).T)
    sa.v.weight.copy_(torch.tensor([[1,0], [0,2]], dtype=torch.float32).T)
x = torch.tensor([[[1,0], [0,1], [1,1]]], dtype=torch.float32)
pad = torch.zeros(1, 3, dtype=torch.bool)
print(sa(x, pad))
print(sa.last)

tensor([[[0.7160, 1.7199],
         [0.5989, 1.6044],
         [0.6943, 1.8514]]], grad_fn=<UnsafeViewBackward0>)
tensor([[[0.1400, 0.2840, 0.5760],
         [0.1978, 0.4011, 0.4011],
         [0.0743, 0.3057, 0.6200]]], grad_fn=<SoftmaxBackward0>)


- H1  Parameter count of TinyTransformer(V = 3332, L = 40, d = 32, ff = 64).  One line each:
      word embedding ; position embedding ; the three attention matrices (no bias) ; two layer
      norms (each has a weight and a bias of d numbers) ; the feed-forward (two Linears with
      biases) ; the output Linear.  Then the total.
      * Embedding: 3332 x 32 = 106,624
      * pos: 40 x 32 = 1,280
      * three attention metrics: 3 x (32 x 32) = 3,072
      * two layer norms: 2 x (2 x 32) = 128
      * feed forward: (32 x 64 + 64) + (64 x 32 + 32) = 4,192
      * out: 32 x 1 + 1 = 33
      * total = 115,329

In [39]:
class Block(nn.Module):
    def __init__(self, d, ff, drop=0.1):
        super().__init__()
        self.att = SelfAttention(d)
        self.n1 = nn.LayerNorm(d)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, ff), nn.ReLU(), nn.Linear(ff, d))
        self.dr = nn.Dropout(drop)

    def forward(self, x, pad):
        x = self.n1(x + self.dr(self.att(x, pad)))
        x = self.n2(x + self.dr(self.ff(x)))
        return x

In [40]:
class TinyTransformer(nn.Module):
    def __init__(self, V, L=40, d=32, ff=64, drop=0.1):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=0)
        self.pos = nn.Embedding(L, d)
        self.block = Block(d, ff, drop)
        self.out = nn.Linear(d, 1)

    def forward(self, x):
        pad = (x == 0)
        h = self.emb(x) + self.pos(torch.arange(x.shape[1]))
        h = self.block(h, pad)
        m = (~pad).unsqueeze(-1).float()
        return self.out((h * m).sum(1) / m.sum(1).clamp(min=1))

In [41]:
print(sum(p.numel() for p in TinyTransformer(len(itos)).parameters()))

115329


In [46]:
def run(seed=42, epochs=60, patience=6):
    torch.manual_seed(seed)
    model = TinyTransformer(len(itos))
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    lf = nn.BCEWithLogitsLoss()
    g = torch.Generator().manual_seed(seed)
    best_vl, best_state, best_ep, bad = 1e9, None, 0, 0
    for ep in  range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), 64):
            idx = perm[i : i + 64]
            opt.zero_grad()
            loss = lf(model(Xt[idx]), yt[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = lf(model(Xv), yv).item()
        if vl < best_vl - 1e-4:
            best_vl, best_state, best_ep, bad = vl, copy.deepcopy(model.state_dict()), ep, 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = (torch.sigmoid(model(Xv)) > 0.5).int().squeeze(1).numpy()
    return model, best_ep, best_vl, f1_score(y_va, p), precision_score(y_va, p), recall_score(y_va, p)

In [47]:
nn_rows = []
for sd in [42, 1, 2]:
    _, be, bl, f1, pr, rc = run(sd)
    nn_rows.append((sd, be, round(bl, 4), round(f1, 4), round(pr, 4), round(rc, 4)))
pd.DataFrame(nn_rows, columns=["seed", "best_epoch", "val_loss", "val_f1", "precision", "recall"])

,seed,best_epoch,val_loss,val_f1,precision,recall
0,42,7,0.0994,0.8947,0.9444,0.8500
1,1,10,0.1079,0.9106,0.9304,0.8917
2,2,8,0.0826,0.9091,0.9459,0.8750


- i expected to transformer to win but it did not.

| Model | val F1 (3 seeds) | val loss
|---|---|---|
|tfidf 1-2gram + LR, C=10|0.9351|n/a
|tfidf + LR balanced, C=10|0.9185|n/a
|AvgEmb|0.906 to 0.920|0.073 to 0.075
|TinyTransformer|0.895 to 0.911|0.083 to 0.108

- Do the AvgEmb and transformer ranges overlap? What does that say?
    * Yes they do overlap. AvgEmb spans 0.906 to 0.920, and TinyTransformer spans 0.895 to 0.911 (overlapping between 0.906 and 0.911)

- Which model has the best validation loss, and what does the loss tell you that F1 does not?
    * AvgEmb has the best validation loss (0.073 to 0.075). F1 is computed after a hard 0.5 cut-off, on about 120 spam messages, so one message flipping moves it by 0.4 points. The loss uses the predicted probabilities of all 892 messages, so it is smoother and less noisy.

- Does the Day 19 choice stand? (Remember: the test set stays closed.)
    * Yes. The Day 19 choice, tfidf + LR balanced (val F1 0.9185), stands.
      The transformer reaches 0.895 to 0.911, which is at or below it, and
      its validation loss (0.083 to 0.108) is worse than AvgEmb's (0.073 to 0.075)
      in all three seeds. It is also more complex (115,329 parameters).

In [48]:
def look(i, k =4):
    x = Xv[i:i + 1]
    with torch.no_grad():
        p = torch.sigmoid(m(x)).item()
    n = int((x != 0).sum())
    toks = tokenise(X_va.iloc[i])[:L]
    recv = m.block.att.last[0][:n, :n].mean(0)
    top = torch.argsort(recv, descending=True)[:k]
    print(round(p, 3), X_va.iloc[i][:70])
    print("    most attended:", [(toks[j], round(recv[j].item(), 3)) for j in top.tolist()],
          "| uniform would be", round(1 / n, 3))

In [49]:
m, *_ = run(42)
spam_ids = [i for i in range(len(X_va)) if y_va.iloc[i] == 1][:3]
ham_ids = [i for i in range(len(X_va)) if y_va.iloc[i] == 0][:2]
for i in spam_ids + ham_ids:
    look(i)

0.989 Todays Voda numbers ending 1225 are selected to receive a £50award. If
    most attended: [('receive', 0.091), ('selected', 0.081), ('code', 0.074), ('3100', 0.069)] | uniform would be 0.038
0.985 FREE for 1st week! No1 Nokia tone 4 ur mobile every week just txt NOKI
    most attended: [('w45wq', 0.095), ('www', 0.088), ('nokia', 0.049), ('mates', 0.048)] | uniform would be 0.03
0.959 Want to funk up ur fone with a weekly new tone reply TONES2U 2 this te
    most attended: [('best', 0.076), ('rates', 0.067), ('co', 0.066), ('text', 0.058)] | uniform would be 0.033
0.002 I want to send something that can sell fast.  &lt;#&gt; k is not easy 
    most attended: [('want', 0.297), ('i', 0.105), ('send', 0.087), ('k', 0.084)] | uniform would be 0.062
0.001 Happy or sad , one thing about past is- "Its no more" GOOD MORNING :-)
    most attended: [('or', 0.221), ('one', 0.151), ('is', 0.097), ('thing', 0.086)] | uniform would be 0.077


**Labels**
- QUESTION: Does a tiny transformer (one self-attention block built by hand) beat my Day 19 models (TF-IDF + logistic regression and the averaging network AvgEmb) at spotting spam in the SMS data?

- CHECK: The same split (random_state 42, an inner validation set of 892 messages with 120 spam), the same vocabulary (3,332 words) and the same length (L = 40). The transformer was trained with 3 seeds (AdamW, lr 1e-3, early stopping with patience 6) and compared on validation F1 and validation loss. The test set was not reopened: it was used once on Day 19, and using it again to choose between models would make it a second validation set. Noise rule: one missed spam is worth 0.0043 F1, so about 3 messages (0.013) is noise. Two checks on the build: my attention layer reproduced the hand-computed attention of Day 20 exactly, and the parameter count from the code (115,329) matched my paper count.

- FINDING: I expected the transformer to win, but it did not. Validation F1: transformer 0.895 to 0.911, AvgEmb 0.906 to 0.920 (the ranges overlap, so a tie on F1), tfidf + LR balanced 0.9185, tfidf 1-2gram 0.9351. Validation loss: transformer 0.083 to 0.108, AvgEmb 0.073 to 0.075, so the transformer is worse in all three seeds. In spam messages the attention gave 2 to 3 times the uniform weight to words of the offer (receive, selected, www, a code, text); in ham messages it concentrated on everyday words (want, or, one). Bug found on the way: one empty validation message made every score `-inf`, the softmax gave `nan`, and the validation loss became `nan`; the fix was `or [1]` in `encode`.

- DECISION: The Day 19 choice, tfidf + LR balanced (C=10), stands, and its test result from Day 19 (F1 0.9485) is unchanged. A transformer pays off with much more data, with pretraining on a huge corpus, with long-range context that matters, or for generating text. That leads to Day 21 (LLMs).

**Questions for notes**
- S20-1  Why does the transformer not clearly beat the simple average on this data?
  * Spam is mostly decided by which words appear (txt, claim, www, free), and an averaged word vector already sees that. The extra context that attention adds matters less here, and 3,565 training messages are too few for a transformer to learn what a pretrained one learned from billions of words.
- S20-2  Why does the pad mask have to be applied INSIDE the attention?
  * The padding mask must be applied inside the attention mechanism (right before the Softmax function) so that real words do not waste attention weight on blank padding tokens. (In H3, without the mask 16.5% of the weight went to the two PAD columns, and that share changes with the padding length. An average at the end cannot undo it, because the pad vectors have already been mixed in.)
- S20-3  What does the residual connection do for training, and what does LayerNorm do for each token?
  * The residual connection (x + sublayer(x)) keeps the original signal and gives the gradients a direct path, which makes stacks of blocks trainable. LayerNorm standardises each token's own d numbers (a z-score per token, then a learned scale and shift), so the scale stays stable from block to block.
- S20-4  What do the position vectors add here, and what did removing them do?
  * They make the input depend on the order of the words; without them the transformer sees a bag of words (the shuffle test: True). In the reference run of the optional experiment (Step 7, which I skipped), removing them lowered F1 by about 3 points, so order helps even for spam ("call now", "txt to claim").
- S20-5  Why did we not reopen the test set today?
  * It was already used once on Day 19, to judge the model we had chosen. Using it again to choose between models would turn it into a second validation set and make the final number optimistic, so today's comparison used the validation split only.
- S20-6  What would make a transformer worth its cost, and where does that lead (Day 21)?
  * A lot of data or, better, pretraining on a huge corpus (the model arrives already knowing language), long-range context that matters, and generation tasks. The cost grows with the square of the length (10,000 times more work for 100 times more tokens). A transformer pretrained on a huge corpus and used to generate text is a large language model: Day 21.